# 06: Report Generation — Complete Chart Gallery

This notebook demonstrates every chart type available in `siege_utilities.reporting.ChartGenerator`.

**Sections:**

1. [Imports & Setup](#1-imports--setup)
2. [Standard Charts](#2-standard-charts) — bar, line, pie, scatter, heatmap
3. [Geographic Maps (Folium)](#3-geographic-maps-folium) — marker, heatmap, cluster, flow
4. [Geographic Maps (Matplotlib)](#4-geographic-maps-matplotlib) — advanced choropleth, bivariate choropleth
5. [3D Visualization](#5-3d-visualization) — 3D scatter/surface
6. [Text-Based Charts](#6-text-based-charts) — proportional bar, heatmap text
7. [Composite Charts & Layout](#7-composite-charts--layout) — dashboard, summary, custom, dispatcher
8. [Complete PDF Report](#8-complete-pdf-report) — multi-page PDF with chart_with_caption, chart_section
9. [PowerPoint Generation](#9-powerpoint-generation)

## Prerequisites

```bash
pip install -e /path/to/siege_utilities[geo]
pip install reportlab python-pptx pillow folium
```

In [ ]:
# 1. Imports & Setup
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML, Image as IPImage
import io
import base64

import siege_utilities as su
su.configure_shared_logging(level="INFO")

from siege_utilities.reporting.chart_generator import ChartGenerator
from reportlab.platypus import Image as RLImage

# --- Path configuration ---
# All downloaded data (shapefiles, etc.) goes here.
# Ensure the directory exists before any data-loading cell runs.
from siege_utilities.config.user_config import get_user_config

DATA_DIR = Path(get_user_config().get_download_directory())
DATA_DIR.mkdir(parents=True, exist_ok=True)

su.log_info(f"Data directory: {DATA_DIR}  (exists={DATA_DIR.exists()})")


def show_rl_image(rl_img):
    """Display a ReportLab Image object in the notebook.

    ChartGenerator methods return ReportLab Image objects whose 'filename'
    attribute is a base64 data URI.  This helper decodes that URI into
    raw PNG bytes so IPython can render it inline.
    """
    if hasattr(rl_img, 'filename') and isinstance(rl_img.filename, str) and rl_img.filename.startswith('data:image'):
        b64_data = rl_img.filename.split(',', 1)[1]
        return IPImage(data=base64.b64decode(b64_data))
    return rl_img


chart_gen = ChartGenerator()
np.random.seed(42)

su.log_info("ChartGenerator initialized")
su.log_info(f"Available create_* methods: {len([m for m in dir(chart_gen) if m.startswith('create_')])}")

In [ ]:
# Sample Data — used throughout the notebook

# 1. Sales data (bar, line, pie, dashboard, PDF)
sales_data = pd.DataFrame({
    'Month': ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun'],
    'Revenue': [45000, 52000, 48000, 61000, 55000, 67000],
    'Expenses': [32000, 35000, 33000, 38000, 36000, 40000],
    'Units': [150, 175, 160, 205, 185, 225],
})
sales_data['Profit'] = sales_data['Revenue'] - sales_data['Expenses']

# 2. Regional data (pie, scatter, text charts)
regional_data = pd.DataFrame({
    'Region': ['North', 'South', 'East', 'West', 'Central'],
    'Sales': [125000, 98000, 145000, 112000, 89000],
    'Customers': [450, 380, 520, 410, 340],
    'Avg Order': [278, 258, 279, 273, 262],
})

# 3. Point data — US cities (marker map, heatmap map, cluster map, 3D)
cities = [
    ('New York', 40.71, -74.01), ('Los Angeles', 34.05, -118.24),
    ('Chicago', 41.88, -87.63), ('Houston', 29.76, -95.37),
    ('Phoenix', 33.45, -112.07), ('Philadelphia', 39.95, -75.17),
    ('San Antonio', 29.42, -98.49), ('San Diego', 32.72, -117.16),
    ('Dallas', 32.78, -96.80), ('San Jose', 37.34, -121.89),
    ('Austin', 30.27, -97.74), ('Jacksonville', 30.33, -81.66),
    ('San Francisco', 37.77, -122.42), ('Columbus', 39.96, -82.99),
    ('Indianapolis', 39.77, -86.16), ('Fort Worth', 32.76, -97.33),
    ('Charlotte', 35.23, -80.84), ('Seattle', 47.61, -122.33),
    ('Denver', 39.74, -104.98), ('Nashville', 36.16, -86.78),
]
point_data = pd.DataFrame(cities, columns=['name', 'lat', 'lon'])
point_data['value'] = np.random.randint(100, 5000, size=len(point_data))
point_data['category'] = np.random.choice(['Tech', 'Finance', 'Healthcare'], size=len(point_data))

# 4. Flow data — origin/destination pairs
flow_data = pd.DataFrame({
    'origin_lat':  [40.71, 34.05, 41.88, 29.76, 47.61, 39.74, 37.77, 33.45],
    'origin_lon':  [-74.01, -118.24, -87.63, -95.37, -122.33, -104.98, -122.42, -112.07],
    'dest_lat':    [34.05, 41.88, 29.76, 33.45, 37.77, 40.71, 47.61, 39.74],
    'dest_lon':    [-118.24, -87.63, -95.37, -112.07, -122.42, -74.01, -122.33, -104.98],
    'flow_value':  [1500, 2200, 800, 1100, 1800, 950, 1300, 600],
})

# 5. Correlation data (heatmap, scatter, summary charts)
n = 30
correlation_data = pd.DataFrame({
    'x': np.random.randn(n) * 10 + 50,
    'y': np.random.randn(n) * 15 + 70,
    'z': np.random.randn(n) * 5 + 20,
    'w': np.random.randn(n) * 8 + 40,
    'v': np.random.randn(n) * 12 + 60,
})

# 6. Heatmap text data — quarterly sales by region
heatmap_text_data = pd.DataFrame([
    {'quarter': 'Q1', 'region': 'North', 'sales': 120},
    {'quarter': 'Q1', 'region': 'South', 'sales': 95},
    {'quarter': 'Q1', 'region': 'East', 'sales': 140},
    {'quarter': 'Q1', 'region': 'West', 'sales': 110},
    {'quarter': 'Q2', 'region': 'North', 'sales': 135},
    {'quarter': 'Q2', 'region': 'South', 'sales': 105},
    {'quarter': 'Q2', 'region': 'East', 'sales': 155},
    {'quarter': 'Q2', 'region': 'West', 'sales': 125},
    {'quarter': 'Q3', 'region': 'North', 'sales': 150},
    {'quarter': 'Q3', 'region': 'South', 'sales': 115},
    {'quarter': 'Q3', 'region': 'East', 'sales': 165},
    {'quarter': 'Q3', 'region': 'West', 'sales': 130},
    {'quarter': 'Q4', 'region': 'North', 'sales': 160},
    {'quarter': 'Q4', 'region': 'South', 'sales': 125},
    {'quarter': 'Q4', 'region': 'East', 'sales': 180},
    {'quarter': 'Q4', 'region': 'West', 'sales': 145},
])

su.log_info(f"sales_data:       {sales_data.shape}")
su.log_info(f"regional_data:    {regional_data.shape}")
su.log_info(f"point_data:       {point_data.shape}")
su.log_info(f"flow_data:        {flow_data.shape}")
su.log_info(f"correlation_data: {correlation_data.shape}")
su.log_info(f"heatmap_text_data:{heatmap_text_data.shape}")

## 2. Standard Charts

Matplotlib-backed charts that return ReportLab `Image` objects.  We use `show_rl_image()` to
decode the base64 data URI for notebook display.

In [ ]:
# 2a. Bar Chart — vertical
img = chart_gen.create_bar_chart(
    sales_data, x_column='Month', y_column='Revenue',
    title='Monthly Revenue',
)
display(show_rl_image(img))

# 2a. Bar Chart — horizontal
img_h = chart_gen.create_bar_chart(
    sales_data, x_column='Month', y_column='Revenue',
    title='Monthly Revenue (Horizontal)',
    chart_type='horizontal',
)
display(show_rl_image(img_h))

In [ ]:
# 2b. Line Chart — multi-series
img = chart_gen.create_line_chart(
    sales_data, x_column='Month',
    y_columns=['Revenue', 'Expenses'],
    title='Revenue vs Expenses',
)
display(show_rl_image(img))

In [ ]:
# 2c. Pie Chart — with legend table
img = chart_gen.create_pie_chart(
    regional_data,
    labels_column='Region', values_column='Sales',
    title='Sales by Region',
)
display(show_rl_image(img))

In [ ]:
# 2d. Scatter Plot — with color coding
img = chart_gen.create_scatter_plot(
    correlation_data,
    x_column='x', y_column='y', color_column='z',
    title='Scatter with Color Coding',
)
display(show_rl_image(img))

In [ ]:
# 2e. Heatmap — correlation matrix
img = chart_gen.create_heatmap(
    correlation_data,
    title='Correlation Matrix',
)
display(show_rl_image(img))

## 3. Geographic Maps (Folium)

Folium-based methods save HTML files and return placeholder ReportLab images.
We call the ChartGenerator API to exercise it, then also create Folium map objects
directly so we get interactive output in the notebook.

In [ ]:
# 3a. Marker Map
import folium
from folium import plugins

# Call ChartGenerator API (saves HTML, returns placeholder)
placeholder = chart_gen.create_marker_map(
    point_data, latitude_column='lat', longitude_column='lon',
    value_column='value', label_column='name',
    title='US Cities - Marker Map', zoom_level=4,
)
su.log_info("create_marker_map() returned placeholder image")

# Display interactive Folium map inline
m = folium.Map(location=[39.0, -98.0], zoom_start=4)
for _, row in point_data.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=max(5, row['value'] / 200),
        popup=f"<b>{row['name']}</b><br>Value: {row['value']:,}",
        color='red', fill=True, fill_opacity=0.7,
    ).add_to(m)
display(m)

In [ ]:
# 3b. Heatmap Map
placeholder = chart_gen.create_heatmap_map(
    point_data, latitude_column='lat', longitude_column='lon',
    value_column='value',
    title='US Cities - Heatmap',
)
su.log_info("create_heatmap_map() returned placeholder image")

# Display interactive heatmap
m = folium.Map(location=[39.0, -98.0], zoom_start=4, tiles='cartodbpositron')
heat_data = [[row['lat'], row['lon'], row['value']] for _, row in point_data.iterrows()]
plugins.HeatMap(heat_data, radius=25, blur=15).add_to(m)
display(m)

In [ ]:
# 3c. Cluster Map
placeholder = chart_gen.create_cluster_map(
    point_data, latitude_column='lat', longitude_column='lon',
    label_column='name',
    title='US Cities - Cluster Map',
)
su.log_info("create_cluster_map() returned placeholder image")

# Display interactive cluster map
m = folium.Map(location=[39.0, -98.0], zoom_start=4, tiles='cartodbpositron')
mc = plugins.MarkerCluster()
for _, row in point_data.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=f"<b>{row['name']}</b><br>Category: {row['category']}",
    ).add_to(mc)
mc.add_to(m)
display(m)

In [ ]:
# 3d. Flow Map
placeholder = chart_gen.create_flow_map(
    flow_data,
    origin_lat_column='origin_lat', origin_lon_column='origin_lon',
    dest_lat_column='dest_lat', dest_lon_column='dest_lon',
    flow_value_column='flow_value',
    title='US City Flows',
)
su.log_info("create_flow_map() returned placeholder image")

# Display interactive flow map
m = folium.Map(location=[39.0, -98.0], zoom_start=4, tiles='cartodbpositron')
max_flow = flow_data['flow_value'].max()
for _, row in flow_data.iterrows():
    weight = max(1, int(row['flow_value'] / max_flow * 8))
    folium.PolyLine(
        locations=[
            [row['origin_lat'], row['origin_lon']],
            [row['dest_lat'], row['dest_lon']],
        ],
        weight=weight, color='red', opacity=0.6,
        popup=f"Flow: {row['flow_value']:,}",
    ).add_to(m)
    folium.CircleMarker([row['origin_lat'], row['origin_lon']], radius=4, color='blue', fill=True).add_to(m)
    folium.CircleMarker([row['dest_lat'], row['dest_lon']], radius=4, color='green', fill=True).add_to(m)
display(m)

## 4. Geographic Maps (Matplotlib & Folium Choropleth)

These methods require a GeoDataFrame with geometry. We load California counties using the
same `get_census_boundaries()` pattern from NB05, then use them for both Folium choropleth
maps and matplotlib-rendered maps.

In [ ]:
# Load CA county boundaries
# DATA_DIR was created in cell 1 — get_census_boundaries() caches files there.
from siege_utilities.geo import get_census_boundaries

counties = get_census_boundaries(year=2020, geographic_level='county', state_fips='06')

if counties is None:
    su.log_warning(f"get_census_boundaries() returned None. Cache dir: {DATA_DIR}")
    su.log_warning("Retrying with year=2021...")
    counties = get_census_boundaries(year=2021, geographic_level='county', state_fips='06')

if counties is None:
    raise RuntimeError(
        f"Could not download Census county boundaries.\n"
        f"  Data directory: {DATA_DIR}\n"
        f"  Check network connectivity and try again, or run NB05 first to cache the shapefile."
    )

ca_counties = counties[counties['statefp'] == '06'].copy()

# Simulate data columns
ca_counties['population'] = np.random.randint(10000, 10000000, size=len(ca_counties))
ca_counties['median_income'] = np.random.randint(40000, 150000, size=len(ca_counties))

su.log_info(f"Loaded {len(ca_counties)} California counties")
su.log_info(f"Columns: {list(ca_counties.columns)[:8]}...")

In [ ]:
# 4a. Choropleth Map (Folium) — uses get_census_boundaries() GeoDataFrame
# Prepare data for Folium choropleth — needs a plain DataFrame + separate GeoJSON
choropleth_data = ca_counties[['geoid', 'population']].copy()

placeholder = chart_gen.create_choropleth_map(
    choropleth_data, geo_data=ca_counties,
    location_column='geoid', value_column='population',
    title='CA Population Choropleth (Folium)',
    map_type='us',
    key_on='feature.properties.geoid',
    fill_color='YlOrRd',
)
su.log_info("create_choropleth_map() — HTML saved, placeholder returned")

# Display interactive Folium map inline
import json
m = folium.Map(location=[37.0, -120.0], zoom_start=6, tiles='cartodbpositron')
folium.Choropleth(
    geo_data=json.loads(ca_counties.to_json()),
    data=choropleth_data,
    columns=['geoid', 'population'],
    key_on='feature.properties.geoid',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name='Population',
).add_to(m)
display(m)

In [ ]:
# 4b. Bivariate Choropleth (Folium) — interactive version of the matplotlib bivariate
placeholder = chart_gen.create_bivariate_choropleth(
    ca_counties, geo_data=ca_counties,
    location_column='geoid',
    value_column1='population', value_column2='median_income',
    title='Population vs Income (Folium)',
    color_scheme='default',
)
su.log_info("create_bivariate_choropleth() — HTML saved, placeholder returned")

# The saved HTML at ~/.siege_utilities/temp_bivariate_choropleth.html is interactive.
# Display the saved map inline:
from IPython.display import IFrame
from pathlib import Path
bivar_path = Path.home() / ".siege_utilities" / "temp_bivariate_choropleth.html"
if bivar_path.exists():
    display(IFrame(src=str(bivar_path), width=800, height=500))

In [ ]:
# 4c. Advanced Choropleth (matplotlib)
img = chart_gen.create_advanced_choropleth(
    ca_counties, geodata=ca_counties,
    location_column='geoid', value_column='population',
    title='CA Population by County (Quantiles)',
    classification='quantiles', bins=5,
)
display(show_rl_image(img))

In [ ]:
# 4d. Bivariate Choropleth (matplotlib)
img = chart_gen.create_bivariate_choropleth_matplotlib(
    ca_counties, geodata=ca_counties,
    location_column='geoid',
    value_column1='population', value_column2='median_income',
    title='Population vs Income (matplotlib)',
)
display(show_rl_image(img))

## 5. 3D Visualization

Uses matplotlib's 3D projection for point-cloud and surface plots.

In [ ]:
# 5a. 3D Map — point cloud with simulated elevation
img = chart_gen.create_3d_map(
    point_data,
    latitude_column='lat', longitude_column='lon',
    elevation_column='value',
    title='3D US City Point Cloud',
)
display(show_rl_image(img))

## 6. Text-Based Charts

These return HTML strings instead of images — useful for email reports, lightweight dashboards,
or environments without matplotlib.

In [ ]:
# 6a. Proportional Text Bar Chart
text_chart = chart_gen.create_proportional_text_bar_chart(
    regional_data,
    labels_column='Region', values_column='Sales',
    title='Sales by Region (Text)',
)
display(HTML(f'<pre style="font-family: monospace; line-height: 1.4;">{text_chart}</pre>'))

In [ ]:
# 6b. Heatmap Text Chart
text_heatmap = chart_gen.create_heatmap_text_chart(
    heatmap_text_data,
    x_column='quarter', y_column='region', value_column='sales',
    title='Quarterly Sales by Region (Text)',
)
display(HTML(f'<pre style="font-family: monospace; line-height: 1.4;">{text_heatmap}</pre>'))

## 7. Composite Charts & Layout

Higher-level methods that combine multiple charts or dispatch by configuration.

In [ ]:
# 7a. Dashboard — 2x2 grid of four chart types
months = sales_data['Month'].tolist()
revenue = sales_data['Revenue'].tolist()
expenses = sales_data['Expenses'].tolist()

dashboard_charts = [
    {
        'type': 'bar',
        'title': 'Revenue by Month',
        'data': {'labels': months, 'datasets': [{'data': revenue}]},
    },
    {
        'type': 'line',
        'title': 'Revenue Trend',
        'data': {'labels': months, 'datasets': [{'data': revenue}]},
    },
    {
        'type': 'pie',
        'title': 'Regional Sales',
        'data': {'labels': regional_data['Region'].tolist(), 'data': regional_data['Sales'].tolist()},
    },
    {
        'type': 'scatter',
        'title': 'X vs Y',
        'data': {'x': correlation_data['x'].tolist(), 'y': correlation_data['y'].tolist()},
    },
]

img = chart_gen.create_dashboard(dashboard_charts, layout='2x2')
display(show_rl_image(img))

In [ ]:
# 7b. DataFrame Summary Charts — auto-histogram for numeric columns
img = chart_gen.create_dataframe_summary_charts(
    correlation_data,
    title='Data Distributions',
)
display(show_rl_image(img))

In [ ]:
# 7c. Custom Chart Dispatcher — create_custom_chart()
config = {
    'type': 'bar',
    'data': sales_data,
    'title': 'Custom Bar Chart via Dispatcher',
}
img = chart_gen.create_custom_chart(config)
display(show_rl_image(img))

In [ ]:
# 7d. generate_chart_from_dataframe — generic dispatcher
for chart_type in ['bar', 'line', 'pie', 'scatter']:
    su.log_info(f"Generating {chart_type} chart...")
    img = chart_gen.generate_chart_from_dataframe(
        sales_data, chart_type=chart_type,
        x_column='Month', y_columns=['Revenue'],
        title=f'{chart_type.title()} via Dispatcher',
    )
    display(show_rl_image(img))

# Heatmap uses correlation_data (all-numeric) — dispatcher computes correlation matrix
su.log_info("Generating heatmap chart...")
img = chart_gen.generate_chart_from_dataframe(
    correlation_data, chart_type='heatmap',
    title='Heatmap via Dispatcher',
)
display(show_rl_image(img))

## 8. Complete PDF Report

Build a multi-page PDF demonstrating `BaseReportTemplate`, `create_chart_with_caption()`,
and `create_chart_section()` — these return ReportLab flowable lists that only make sense
inside a PDF build pipeline.

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
from reportlab.lib.units import inch
from siege_utilities.reporting.templates.base_template import BaseReportTemplate

output_pdf = '/tmp/complete_chart_gallery.pdf'

template = BaseReportTemplate(
    output_filename=output_pdf,
    page_size='letter',
    margins={'top': 1.0, 'bottom': 1.0, 'left': 1.0, 'right': 1.0},
)
styles = getSampleStyleSheet()
story = []

# --- Title page ---
story.extend(template.add_title_page(
    title='Chart Gallery Report',
    subtitle='Every ChartGenerator chart type in one PDF',
    author='siege_utilities NB06',
    date='2026',
))

# --- Section: Bar chart with caption ---
bar_img = chart_gen.create_bar_chart(
    sales_data, x_column='Month', y_column='Revenue',
    title='Monthly Revenue', width=5.0, height=3.0,
)
story.extend(chart_gen.create_chart_with_caption(
    bar_img, 'Figure 1: Monthly revenue showing steady growth through H1 2026.',
))

# --- Section: Multi-chart section ---
line_img = chart_gen.create_line_chart(
    sales_data, x_column='Month', y_columns=['Revenue', 'Expenses'],
    title='Revenue vs Expenses', width=5.0, height=3.0,
)
pie_img = chart_gen.create_pie_chart(
    regional_data, labels_column='Region', values_column='Sales',
    title='Regional Sales', width=5.0, height=3.0,
)
story.extend(chart_gen.create_chart_section(
    title='Financial Overview',
    charts=[line_img, pie_img],
    description='Revenue vs expenses trend and regional sales distribution.',
))

# --- Dashboard page ---
story.append(PageBreak())
story.append(Paragraph('Dashboard', styles['Heading1']))
story.append(Spacer(1, 12))

dashboard_img = chart_gen.create_dashboard(dashboard_charts, layout='2x2', width=7.0, height=5.0)
story.extend(chart_gen.create_chart_with_caption(
    dashboard_img, 'Figure 4: Four-panel dashboard with bar, line, pie, and scatter charts.',
))

# Build PDF
template.build_document(story)
su.log_info(f"PDF written to {output_pdf}")

## 9. PowerPoint Generation

In [ ]:
try:
    from pptx import Presentation
    from pptx.util import Inches, Pt
    from datetime import datetime

    prs = Presentation()

    # Title slide
    slide = prs.slides.add_slide(prs.slide_layouts[0])
    slide.shapes.title.text = 'Sales Performance Review'
    slide.placeholders[1].text = f'H1 2026 | Generated {datetime.now().strftime("%Y-%m-%d")}'

    # Data table slide
    slide = prs.slides.add_slide(prs.slide_layouts[5])
    slide.shapes.title.text = 'Sales Summary'
    rows, cols = len(sales_data) + 1, len(sales_data.columns)
    table = slide.shapes.add_table(rows, cols, Inches(0.5), Inches(1.5), Inches(9), Inches(0.8 * rows)).table
    for i, col_name in enumerate(sales_data.columns):
        table.cell(0, i).text = str(col_name)
    for row_idx, row in sales_data.iterrows():
        for col_idx, value in enumerate(row):
            table.cell(row_idx + 1, col_idx).text = str(value)

    pptx_path = '/tmp/sales_presentation.pptx'
    prs.save(pptx_path)
    su.log_info(f"PowerPoint created: {pptx_path}")

except ImportError:
    su.log_warning("python-pptx not installed. Run: pip install python-pptx")

## Method Reference

Complete inventory of `ChartGenerator.create_*` methods — all 21 demonstrated above.

| # | Method | Backend | Returns | Section |
|---|--------|---------|---------|--------|
| 1 | `create_bar_chart` | matplotlib | RL Image | 2a |
| 2 | `create_line_chart` | matplotlib | RL Image | 2b |
| 3 | `create_pie_chart` | matplotlib | RL Image | 2c |
| 4 | `create_scatter_plot` | matplotlib | RL Image | 2d |
| 5 | `create_heatmap` | seaborn | RL Image | 2e |
| 6 | `create_choropleth_map` | Folium | Placeholder | 4a |
| 7 | `create_bivariate_choropleth` | Folium + gpd | Placeholder | 4b |
| 8 | `create_bivariate_choropleth_matplotlib` | matplotlib + gpd | RL Image | 4d |
| 9 | `create_advanced_choropleth` | matplotlib + gpd | RL Image | 4c |
| 10 | `create_marker_map` | Folium | Placeholder | 3a |
| 11 | `create_3d_map` | matplotlib 3D | RL Image | 5a |
| 12 | `create_heatmap_map` | Folium | Placeholder | 3b |
| 13 | `create_cluster_map` | Folium | Placeholder | 3c |
| 14 | `create_flow_map` | Folium | Placeholder | 3d |
| 15 | `create_proportional_text_bar_chart` | text | HTML string | 6a |
| 16 | `create_heatmap_text_chart` | text | HTML string | 6b |
| 17 | `create_dashboard` | matplotlib | RL Image | 7a |
| 18 | `create_dataframe_summary_charts` | matplotlib | RL Image | 7b |
| 19 | `create_chart_with_caption` | ReportLab | List[flowable] | 8 |
| 20 | `create_chart_section` | ReportLab | List[flowable] | 8 |
| 21 | `create_custom_chart` | dispatcher | RL Image | 7c |
| -- | `generate_chart_from_dataframe` | dispatcher | RL Image | 7d |

**Notes:**
- Methods 19 & 20 (`create_chart_with_caption`, `create_chart_section`) return ReportLab flowable lists — only usable inside a PDF build pipeline (Section 8).

**Files Generated:**
- `/tmp/complete_chart_gallery.pdf` — Multi-page PDF with captioned charts
- `/tmp/sales_presentation.pptx` — PowerPoint deck
- `~/.siege_utilities/temp_choropleth_map.html` — Folium choropleth
- `~/.siege_utilities/temp_bivariate_choropleth.html` — Folium bivariate choropleth